# 04 — Segmentation Results: Qualitative Visualisations

Runs batched inference on the test sets of all 5 folds and produces:

- **Per-fold per-image dice/IoU tables** — `outputs/tables/.../fold_k/fold_k_per_image_dice.csv`
- **Sample prediction figures** (best / worst / random by Dice) — `outputs/figures/.../fold_k/sample_predictions/`
- **Cross-fold summary tables** — `qualitative_fold_summary.csv`, `qualitative_class_summary.csv`, `qualitative_per_image.csv`
- **Dice distribution histogram** — `dice_histogram_by_fold.png`

Reads checkpoints and writes outputs **directly to Drive** (small I/O).  
Reads images from the **local SSD copy** (fast inference).  

**Prerequisites:** run `03_train.ipynb` and sync to Drive before running this notebook.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"REPO_ROOT:  {REPO_ROOT}")

In [ ]:
import os, psutil
print(f"CPU count:   {os.cpu_count()}")
print(f"RAM total:   {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"RAM avail:   {psutil.virtual_memory().available / 1e9:.1f} GB")

## Cell 3 — EXPERIMENT to visualize

Edit `RECIPE`, `DATASET`, and `SPLIT_SCHEME` to match the experiment you trained in `03_train.ipynb`.

Visualization knobs: `FOLDS_TO_VIS`, `N_BEST/WORST/RANDOM_PER_FOLD`, `SHOW_INLINE`.

In [ ]:
import torch
from configs.seg.reference_experiments import get_experiment, REFERENCE_RECIPES

# ── What to visualize ──────────────────────────────────────────────────────
RECIPE       = "03_dicebce"      # one of REFERENCE_RECIPES
DATASET      = "figshare"        # "figshare" | "brats2020"
SPLIT_SCHEME = "image_level"    # "image_level"  = §13 reference reproduction (image-level KFold)
                               # "patient_level" = methodologically correct (no patient leakage)

EXPERIMENT = get_experiment(RECIPE, dataset=DATASET, split_scheme=SPLIT_SCHEME, fold=1)

assert EXPERIMENT["task"] == "segmentation"
assert EXPERIMENT["dataset"] in ("figshare", "brats2020")
assert EXPERIMENT["split_scheme"] in ("image_level", "patient_level")

# ── Visualization settings ─────────────────────────────────────────────────
FOLDS_TO_VIS      = [1, 2, 3, 4, 5]  # set [1] first to spot-check
N_BEST_PER_FOLD   = 3
N_WORST_PER_FOLD  = 3
N_RANDOM_PER_FOLD = 3
RANDOM_STATE      = 42
SHOW_INLINE       = True   # False suppresses inline display (faster in long loops)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"experiment   : {EXPERIMENT['name']}")
print(f"dataset      : {EXPERIMENT['dataset']}")
print(f"split_scheme : {EXPERIMENT['split_scheme']}")
print(f"model        : {EXPERIMENT['model_name']}")
print(f"folds        : {FOLDS_TO_VIS}")
print(f"device       : {device}")
print(f"\nAvailable recipes: {REFERENCE_RECIPES}")

## Cell 4 — Copy dataset to local SSD

`copy_to_local` auto-detects a zip at `DRIVE_ROOT/data/<dataset>.zip` and extracts it locally (~2–3 min).  
Falls back to `shutil.copytree` when no zip exists.  

Checkpoints and output figures/tables are read from / written to Drive directly (small files, OK over FUSE).

In [ ]:
from src.train_utils    import set_global_seed
from src.notebook_setup import copy_to_local

set_global_seed(EXPERIMENT["seed"])
LOCAL_ROOT = copy_to_local(DRIVE_ROOT, datasets=[EXPERIMENT["dataset"]])
print(f"LOCAL_ROOT: {LOCAL_ROOT}")

## Cell 5 — `visualize_fold` function

Runs batched inference, saves per-image metrics, and renders best / worst / random prediction figures.

In [ ]:
import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

from src.data_utils      import load_metadata, validate_metadata
from src.sg_data_utils   import BrainTumorDataset, build_eval_transform
from src.sg_test_utils   import load_model_from_pt
from src.sg_metrics      import dice_score, iou_score
from src.sg_vis_utils    import show_image_gt_pred_overlay
from src.file_utils      import experiment_paths, fold_split_csv_paths, experiment_root_paths


@torch.no_grad()
def visualize_fold(fold: int):
    """
    Batched inference on fold `fold`'s test set.

    Writes:
        outputs/tables/<task>/<dataset>/<exp>/fold_<k>/fold_<k>_per_image_dice.csv
        outputs/figures/<task>/<dataset>/<exp>/fold_<k>/sample_predictions/{best,worst,random}_*.png

    Returns the per-image DataFrame, or None if the checkpoint is missing.
    """
    # Drive paths: checkpoints are read from Drive; figures + tables written to Drive.
    drive_paths = experiment_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
        fold=fold,
    )
    # Local paths: fold CSVs were copied with the dataset.
    csv_paths = fold_split_csv_paths(
        LOCAL_ROOT,
        dataset=EXPERIMENT["dataset"],
        scheme=EXPERIMENT["split_scheme"],
        fold=fold,
    )

    pt_path = drive_paths["best_model"]
    if not pt_path.exists():
        print(f"  fold {fold}: best_model.pt not found at {pt_path} — skipping")
        return None

    fig_dir = drive_paths["figures"] / "sample_predictions"
    fig_dir.mkdir(parents=True, exist_ok=True)

    test_df = load_metadata(csv_paths["test"]).reset_index(drop=True)
    validate_metadata(test_df)
    print(f"\n{'='*60}\n  FOLD {fold}  ({len(test_df)} test images)\n{'='*60}")

    model, _ = load_model_from_pt(pt_path=pt_path, device=device)
    eval_tf  = build_eval_transform(
        image_size=EXPERIMENT["image_size"],
        preprocessing=EXPERIMENT["preprocessing"],
    )
    ds     = BrainTumorDataset(test_df, LOCAL_ROOT, transform=eval_tf, return_meta=True)
    loader = DataLoader(ds, batch_size=32, shuffle=False,
                        num_workers=2, pin_memory=False, drop_last=False)

    per_image_records, cached_preds, cached_gts = [], {}, {}
    n_seen = 0
    t0 = time.time()

    for x, y, metas in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        pred_b = (torch.sigmoid(logits) > EXPERIMENT["threshold"]).float()

        d      = dice_score(pred_b, y, reduce="none").cpu().numpy()
        j      = iou_score (pred_b, y, reduce="none").cpu().numpy()
        pred_np = pred_b.cpu().numpy().astype(np.uint8)[:, 0]
        gt_np   = (y.cpu().numpy() > 0).astype(np.uint8)[:, 0]

        for i in range(x.size(0)):
            img_id = metas["image_id"][i]
            row    = test_df.iloc[n_seen + i]
            per_image_records.append({
                "image_id":    img_id,
                "patient_id":  metas["patient_id"][i],
                "tumor_class": metas["tumor_class"][i],
                "image_path":  str(row["image_path"]),
                "mask_path":   str(row["mask_path"]),
                "dice":        float(d[i]),
                "iou":         float(j[i]),
            })
            cached_preds[img_id] = pred_np[i]
            cached_gts[img_id]   = gt_np[i]

        n_seen += x.size(0)
        if n_seen % 128 == 0 or n_seen == len(test_df):
            print(f"    inference {n_seen}/{len(test_df)}  ({time.time()-t0:.1f}s)")

    per_image_df = pd.DataFrame(per_image_records)
    print(f"  dice  mean={per_image_df['dice'].mean():.4f}  "
          f"std={per_image_df['dice'].std():.4f}")
    print(f"  iou   mean={per_image_df['iou'].mean():.4f}")

    # ── Per-image metrics table ────────────────────────────────────────────
    table_csv = drive_paths["tables"] / f"fold_{fold}_per_image_dice.csv"
    per_image_df.to_csv(table_csv, index=False)
    print(f"  table  -> {table_csv.relative_to(DRIVE_ROOT)}")

    # ── Best / worst / random selection ───────────────────────────────────
    best   = per_image_df.nlargest (N_BEST_PER_FOLD,   "dice")
    worst  = per_image_df.nsmallest(N_WORST_PER_FOLD,  "dice")
    rng    = per_image_df.sample(
        n=min(N_RANDOM_PER_FOLD, len(per_image_df)), random_state=RANDOM_STATE,
    )

    for kind, rows in {"best": best, "worst": worst, "random": rng}.items():
        for rank, (_, row) in enumerate(rows.iterrows(), start=1):
            img_id = row["image_id"]
            img    = cv2.imread(str(LOCAL_ROOT / row["image_path"]), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(
                    img,
                    (EXPERIMENT["image_size"], EXPERIMENT["image_size"]),
                    interpolation=cv2.INTER_LINEAR,
                )
            else:
                img = np.zeros(
                    (EXPERIMENT["image_size"], EXPERIMENT["image_size"]), np.uint8
                )

            show_image_gt_pred_overlay(
                image     = img,
                gt_mask   = cached_gts[img_id],
                pred_mask = cached_preds[img_id],
                title     = (
                    f"fold {fold} | {kind} #{rank} | id={img_id} "
                    f"| class={row['tumor_class']} | patient={row['patient_id']}"
                ),
                dice      = row["dice"],
                iou       = row["iou"],
                save_path = fig_dir / f"{kind}_{rank:02d}_id{img_id}.png",
                show      = SHOW_INLINE,
            )

    print(f"  figures -> {fig_dir.relative_to(DRIVE_ROOT)}")
    print(f"  fold {fold} done in {time.time()-t0:.1f}s")
    return per_image_df


print("visualize_fold defined.")

## Cell 6 — Run visualizations

Iterates over `FOLDS_TO_VIS`. Each fold: loads checkpoint, runs inference, saves figures + table.

In [ ]:
fold_dfs = {}
for fold in FOLDS_TO_VIS:
    fold_dfs[fold] = visualize_fold(fold)

ok_folds = [k for k, v in fold_dfs.items() if v is not None]
print(f"\nDone. {len(ok_folds)}/{len(FOLDS_TO_VIS)} folds visualized: {ok_folds}")

## Cell 7 — Cross-fold summary tables

Aggregates per-fold and per-class Dice / IoU across all visualized folds and writes three CSVs to
`outputs/tables/<task>/<dataset>/<exp>/`.

In [ ]:
from IPython.display import display

all_dfs = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs:
    print("No fold results to summarize.")
else:
    combined = pd.concat(all_dfs, ignore_index=True)

    fold_summary  = (
        combined.groupby("fold")[["dice", "iou"]]
        .agg(["mean", "std", "count"])
        .round(4)
    )
    class_summary = (
        combined.groupby("tumor_class")[["dice", "iou"]]
        .agg(["mean", "std", "count"])
        .round(4)
    )

    print("Per-fold Dice / IoU:")
    display(fold_summary)
    print("\nPer-class Dice / IoU (pooled across folds):")
    display(class_summary)

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_dir = root_paths["tables"]
    fold_summary.to_csv (out_dir / "qualitative_fold_summary.csv")
    class_summary.to_csv(out_dir / "qualitative_class_summary.csv")
    combined.to_csv     (out_dir / "qualitative_per_image.csv",   index=False)
    print(f"\nSaved 3 CSVs -> {out_dir.relative_to(DRIVE_ROOT)}")

## Cell 8 — Dice distribution histograms

One histogram per fold, showing the spread of per-image Dice scores.  
Saved as `outputs/figures/<task>/<dataset>/<exp>/dice_histogram_by_fold.png`.

In [ ]:
import matplotlib.pyplot as plt

all_dfs = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs:
    print("No fold results to plot.")
else:
    ok_list = [(fold, df) for fold, df in fold_dfs.items() if df is not None]
    n = len(ok_list)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, (fold, df) in zip(axes, ok_list):
        ax.hist(df["dice"].values, bins=20, range=(0, 1),
                color="#2196F3", edgecolor="white", alpha=0.85)
        ax.axvline(df["dice"].mean(), linestyle="--", color="#f44336",
                   label=f"mean={df['dice'].mean():.3f}")
        ax.set_xlabel("Dice score")
        ax.set_title(f"fold {fold}")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)

    axes[0].set_ylabel("images (count)")
    fig.suptitle(
        f"{EXPERIMENT['name']} \u00b7 {EXPERIMENT['dataset']} "
        f"\u2014 Dice distribution by fold"
    )
    fig.tight_layout()

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_png = root_paths["figures"] / "dice_histogram_by_fold.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"saved {out_png.relative_to(DRIVE_ROOT)}")